# Demo 01 : Interrupts

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · interrupt

We port the write, review and send workflow from ARI510 demo22 to a Veenhaven Bus work order. Sending records a proposal in state and makes no external change.

In [2]:
import json
import operator
import sqlite3
import time
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Usage(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.calls += 1

    def on_llm_end(self, response, **kwargs):
        for batch in response.generations:
            for generation in batch:
                usage = generation.message.usage_metadata or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)

usage = Usage()
llm = make_llm(callbacks=[usage], max_retries=0, max_tokens=650)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as conn:
    conn.row_factory = sqlite3.Row
    work_order = dict(conn.execute("SELECT * FROM work_orders WHERE wo_id = ?", ("BW-7001",)).fetchone())
    part = dict(conn.execute("SELECT * FROM parts WHERE part_no = ?", ("CE-CF-02",)).fetchone())
print("work order:", work_order)
print("part:", part)

def config(label):
    return {"configurable": {"thread_id": f"07-{label}-{uuid4()}"}, "recursion_limit": 30}

work order: {'wo_id': 'BW-7001', 'fleet_no': 'CE12-2021-0117', 'opened': '2026-09-14', 'status': 'open', 'error_code': 'F-CE-110', 'summary': 'Traction system over-temperature during afternoon route', 'mechanic_id': 'M01'}
part: {'part_no': 'CE-CF-02', 'model': 'CE-12', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}


In [3]:
class Reply(TypedDict):
    complaint: str
    draft: str
    feedback: str
    sent: str

def write(state: Reply):
    draft = llm.invoke(
        "Write a short internal Veenhaven Bus repair proposal. Do not invent a diagnosis. "
        f"Work order: {state['complaint']}. Available part: {part}. "
        f"Reviewer feedback: {state['feedback']}"
    ).content
    return {"draft": draft}

def review(state: Reply) -> Command[Literal["send", "write"]]:
    decision = interrupt({"draft": state["draft"], "options": ["approve", "edit", "reject"]})
    if decision["action"] == "approve":
        return Command(goto="send")
    if decision["action"] == "edit":
        return Command(update={"draft": decision["text"]}, goto="send")
    if decision["action"] == "reject":
        return Command(update={"feedback": decision["text"]}, goto="write")
    raise ValueError("Unknown review action")

def send(state: Reply):
    return {"sent": state["draft"]}

builder = StateGraph(Reply)
builder.add_node("write", write)
builder.add_node("review", review)
builder.add_node("send", send)
builder.add_edge(START, "write")
builder.add_edge("write", "review")
builder.add_edge("send", END)
graph = builder.compile(checkpointer=InMemorySaver())
inputs = {"complaint": json.dumps(work_order), "draft": "", "feedback": "", "sent": ""}

In [4]:
cfg = config("review")
paused = graph.invoke(inputs, cfg, version="v2")
print("payload:", paused.interrupts[0].value)
print("next:", graph.get_state(cfg).next)
print("sent:", paused.value["sent"])
assert paused.interrupts and not paused.value["sent"]
approved = graph.invoke(Command(resume={"action": "approve"}), cfg, version="v2")
print("approved:", approved.value["sent"])
assert not approved.interrupts and approved.value["sent"]

payload: {'draft': '**Internal Repair Proposal — Veenhaven Bus**\n\n**Work order:** BW-7001  \n**Fleet number:** CE12-2021-0117  \n**Opened:** 14 September 2026  \n**Status:** Open  \n**Reported issue:** Traction system over-temperature during afternoon route  \n**Error code:** F-CE-110  \n**Assigned mechanic:** M01\n\n**Proposal:** Inspect and diagnose the traction cooling system. Do not replace parts until the fault is confirmed. If inspection identifies the traction cooling fan as faulty, CE-12 fan **CE-CF-02** is available at the Rietpoort depot (6 in stock) for **€420**.\n\n**Next step:** Record diagnostic findings and obtain approval for any required parts or further work.', 'options': ['approve', 'edit', 'reject']}
next: ('review',)
sent: 
approved: **Internal Repair Proposal — Veenhaven Bus**

**Work order:** BW-7001  
**Fleet number:** CE12-2021-0117  
**Opened:** 14 September 2026  
**Status:** Open  
**Reported issue:** Traction system over-temperature during afternoon route

## b · approve-edit-reject

We supply an edited proposal, then reject another draft with feedback. Rejection returns to writing and asks for approval again. The model may change the wording or retain parts of its earlier proposal.

We reuse the saved draft in fresh threads with `update_state(..., as_node="write")`, from module 03. The pause before writing avoids another model call before each review.

In [5]:
edit_cfg = config("edit")
graph.invoke(inputs, edit_cfg, version="v2", interrupt_before=["write"])
graph.update_state(edit_cfg, {"draft": approved.value["draft"]}, as_node="write")
graph.invoke(None, edit_cfg, version="v2")
edited_text = "Inspect BW-7001 before reserving CE-CF-02. No part order is authorised."
edited = graph.invoke(Command(resume={"action": "edit", "text": edited_text}), edit_cfg, version="v2")
print("edited:", edited.value["sent"])
assert edited.value["sent"] == edited_text

reject_cfg = config("reject")
graph.invoke(inputs, reject_cfg, version="v2", interrupt_before=["write"])
graph.update_state(reject_cfg, {"draft": approved.value["draft"]}, as_node="write")
graph.invoke(None, reject_cfg, version="v2")
rewritten = graph.invoke(Command(resume={"action": "reject", "text": "Require an inspection before selecting a replacement part."}), reject_cfg, version="v2")
print("revised:", rewritten.interrupts[0].value)
assert rewritten.interrupts and not rewritten.value["sent"]
final = graph.invoke(Command(resume={"action": "approve"}), reject_cfg, version="v2")
print("sent:", final.value["sent"])
assert final.value["sent"]

edited: Inspect BW-7001 before reserving CE-CF-02. No part order is authorised.


revised: {'draft': '**Internal Repair Proposal — Veenhaven Bus**\n\n**Work order:** BW-7001 | **Fleet no.:** CE12-2021-0117  \n**Opened:** 14 September 2026 | **Status:** Open  \n**Reported issue:** Traction system over-temperature during afternoon route (F-CE-110)\n\n**Proposal:** Have mechanic M01 inspect the traction system and cooling components to identify the cause. Do not select or authorize a replacement part until the inspection findings are available.\n\n**Potential part, pending inspection:** Traction cooling fan (CE-CF-02), €420.00; 6 in stock at Rietpoort.', 'options': ['approve', 'edit', 'reject']}
sent: **Internal Repair Proposal — Veenhaven Bus**

**Work order:** BW-7001 | **Fleet no.:** CE12-2021-0117  
**Opened:** 14 September 2026 | **Status:** Open  
**Reported issue:** Traction system over-temperature during afternoon route (F-CE-110)

**Proposal:** Have mechanic M01 inspect the traction system and cooling components to identify the cause. Do not select or authoriz

## c · several-interrupts

We keep sequential interrupt calls in a stable order inside one node. We then pause parallel nodes in the same step and match each reply to its interrupt id.

In [6]:
class Checks(TypedDict):
    answers: Annotated[list[str], operator.add]

def sequential(state):
    print("review entered")
    safety = interrupt({"check": "safety", "work_order": work_order["wo_id"]})
    budget = interrupt({"check": "budget", "part": part["part_no"]})
    return {"answers": [f"safety={safety}", f"budget={budget}"]}

seq = StateGraph(Checks).add_node("review", sequential).add_edge(START, "review").add_edge("review", END).compile(checkpointer=InMemorySaver())
seq_cfg = config("sequential")
out = seq.invoke({"answers": []}, seq_cfg, version="v2")
print("first:", out.interrupts[0].value)
out = seq.invoke(Command(resume="inspection required"), seq_cfg, version="v2")
print("second:", out.interrupts[0].value)
out = seq.invoke(Command(resume="approved budget"), seq_cfg, version="v2")
print("answers:", out.value["answers"])
assert len(out.value["answers"]) == 2

def safety(state):
    answer = interrupt({"check": "safety", "work_order": work_order["wo_id"]})
    return {"answers": [f"safety={answer}"]}

def budget(state):
    answer = interrupt({"check": "budget", "part": part["part_no"]})
    return {"answers": [f"budget={answer}"]}

parallel = StateGraph(Checks)
parallel.add_node("safety", safety)
parallel.add_node("budget", budget)
for name in ("safety", "budget"):
    parallel.add_edge(START, name)
    parallel.add_edge(name, END)
checks = parallel.compile(checkpointer=InMemorySaver())
check_cfg = config("parallel")
out = checks.invoke({"answers": []}, check_cfg, version="v2")
print("pending:", [(item.id, item.value) for item in out.interrupts])
replies = {item.id: f"approved {item.value['check']}" for item in out.interrupts}
out = checks.invoke(Command(resume=replies), check_cfg, version="v2")
print("answers:", sorted(out.value["answers"]))
assert len(out.value["answers"]) == 2 and not out.interrupts

review entered
first: {'check': 'safety', 'work_order': 'BW-7001'}
review entered
second: {'check': 'budget', 'part': 'CE-CF-02'}
review entered
answers: ['safety=inspection required', 'budget=approved budget']
pending: [('53fa27a444e0cc0ac151fc6d63ef2ea2', {'check': 'budget', 'part': 'CE-CF-02'}), ('e2aa415e151e975ebca8a7cfad5b92f9', {'check': 'safety', 'work_order': 'BW-7001'})]
answers: ['budget=approved budget', 'safety=approved safety']


A client such as Studio or an A2A caller often answers one interrupt at a time. We put the two checks and a finished `stock` sibling in a subgraph, answer one id per resume and count how often each node body runs.

In [7]:
runs = {"stock": 0, "safety": 0, "budget": 0}

def counted(name, check):
    def node(state):
        if check is None:
            runs[name] += 1
            return {"answers": [f"{name}=read"]}
        answer = interrupt({"check": check})
        runs[name] += 1
        return {"answers": [f"{name}={answer}"]}
    return node

inner = StateGraph(Checks)
for name, check in [("stock", None), ("safety", "safety"), ("budget", "budget")]:
    inner.add_node(name, counted(name, check))
    inner.add_edge(START, name)
    inner.add_edge(name, END)
outer = StateGraph(Checks).add_node("checks", inner.compile()).add_edge(START, "checks").add_edge("checks", END)
one_by_one = outer.compile(checkpointer=InMemorySaver())
one_cfg = config("one-by-one")
out = one_by_one.invoke({"answers": []}, one_cfg, version="v2")
for item in out.interrupts:
    out = one_by_one.invoke(Command(resume={item.id: f"approved {item.value['check']}"}), one_cfg, version="v2")
    print("answered:", item.value["check"], "still pending:", [i.value["check"] for i in out.interrupts], "runs:", runs)
print("answers:", sorted(out.value["answers"]))
assert runs == {"stock": 1, "safety": 1, "budget": 1}

answered: safety still pending: ['budget'] runs: {'stock': 1, 'safety': 1, 'budget': 0}
answered: budget still pending: [] runs: {'stock': 1, 'safety': 1, 'budget': 1}
answers: ['budget=approved budget', 'safety=approved safety', 'stock=read']


In [8]:
print("metrics:", json.dumps({"model": MODEL, "calls": usage.calls,
      "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens,
      "seconds": round(time.perf_counter() - started, 1)}))

metrics: {"model": "openai/gpt-6-luna", "calls": 2, "input_tokens": 330, "output_tokens": 297, "seconds": 7.9}


## Things to try

Optional : we can reject again or change the parallel replies. We keep interrupt order stable during a paused run.